In [ ]:
# Shared repository paths (no model or dataset is loaded here).
from pathlib import Path as _RepositoryPath
import os as _repository_os
import sys as _repository_sys

_repository_hint = _repository_os.environ.get("TESI_REPOSITORY_ROOT")
_repository_starts = (
    [_RepositoryPath(_repository_hint).expanduser().resolve()]
    if _repository_hint else
    ([_RepositoryPath(__file__).resolve().parent] if "__file__" in globals() else [])
    + [_RepositoryPath.cwd().resolve()]
)
_repository_root = next(
    (candidate for start in _repository_starts for candidate in (start, *start.parents)
     if (candidate / "project_paths.py").is_file()), None
)
if _repository_root is None:
    raise FileNotFoundError(
        "Start Jupyter inside the repository or set TESI_REPOSITORY_ROOT to its root."
    )
if str(_repository_root) not in _repository_sys.path:
    _repository_sys.path.insert(0, str(_repository_root))
from project_paths import PROJECT_ROOT as TESI_PROJECT_ROOT
from project_paths import BASELINES_PATH as TESI_BASELINES_PATH


In [ ]:
# =========================
# SETUP PATH
# =========================
import os

BASE_PATH = str(TESI_PROJECT_ROOT / 'Secondo_Esperimento/Preprocessing/Dataset')
RAW_PATH = os.path.join(BASE_PATH, "RawDataset")

PATH_SST = os.path.join(RAW_PATH, "SST")
PATH_MNLI = os.path.join(RAW_PATH, "MNLI")

HF_CACHE = str(TESI_PROJECT_ROOT / 'Secondo_Esperimento/hf_cache')
TMP_PATH = str(TESI_PROJECT_ROOT / 'Secondo_Esperimento/tmp')

os.environ["HF_HOME"] = HF_CACHE
os.environ["HF_DATASETS_CACHE"] = os.path.join(HF_CACHE, "datasets")
os.environ["HF_HUB_CACHE"] = os.path.join(HF_CACHE, "hub")
os.environ["TMPDIR"] = TMP_PATH
os.environ["HF_HUB_DISABLE_XET"] = "1"

from datasets import load_dataset

# =========================
# CREATE FOLDERS
# =========================
os.makedirs(BASE_PATH, exist_ok=True)
os.makedirs(RAW_PATH, exist_ok=True)
os.makedirs(PATH_SST, exist_ok=True)
os.makedirs(PATH_MNLI, exist_ok=True)

os.makedirs(HF_CACHE, exist_ok=True)
os.makedirs(os.path.join(HF_CACHE, "datasets"), exist_ok=True)
os.makedirs(os.path.join(HF_CACHE, "hub"), exist_ok=True)
os.makedirs(TMP_PATH, exist_ok=True)

# =========================
# DOWNLOAD ONLY TRAIN SPLITS
# =========================
ds_sst_train = load_dataset(
    "yassiracharki/Amazon_Reviews_Binary_for_Sentiment_Analysis",
    split="train"
)

ds_mnli_train = load_dataset(
    "ICKD/mnli",
    split="train"
)

# =========================
# SAVE TO DISK
# =========================
ds_sst_train.save_to_disk(PATH_SST)
ds_mnli_train.save_to_disk(PATH_MNLI)

In [ ]:
# =========================
# PRETTY DATASET SUMMARY
# =========================
import pandas as pd
from datasets import load_from_disk

ds_sst = load_from_disk(PATH_SST)
ds_mnli = load_from_disk(PATH_MNLI)

In [ ]:
# =========================
# SST PREPROCESSING
# =========================
ds_sst = ds_sst.remove_columns(
    [col for col in ds_sst.column_names if col not in ["review_text", "class_index"]]
)

ds_sst = ds_sst.rename_columns({
    "review_text": "text",
    "class_index": "label"
})

# =========================
# MNLI PREPROCESSING
# =========================
ds_mnli = ds_mnli.remove_columns(
    [col for col in ds_mnli.column_names if col not in ["text", "label"]]
)

In [ ]:
# =========================
# SST LABEL REMAPPING
# =========================
def remap_sst(example):
    example["label"] = example["label"] - 1
    return example

ds_sst = ds_sst.map(remap_sst)

In [ ]:
# =========================
# MNLI LABEL REMAPPING
# Dataset format:
# 0 = entailment, 1 = neutral, 2 = contradiction
#
# TextAttack MNLI model format:
# 0 = contradiction, 1 = entailment, 2 = neutral
# =========================

ds_mnli = ds_mnli.map(
    lambda example: {
        "label": {
            0: 1,  # entailment -> entailment
            1: 2,  # neutral -> neutral
            2: 0   # contradiction -> contradiction
        }[example["label"]]
    }
)

In [ ]:
# =========================
# CHECK LABELS
# =========================
print("SST unique labels:", sorted(set(ds_sst["label"])))
print("MNLI unique labels:", sorted(set(ds_mnli["label"])))

assert set(ds_sst["label"]) == {0, 1}, "SST labels non sono {0,1}"
assert set(ds_mnli["label"]) == {0, 1, 2}, "MNLI labels non sono {0,1,2}"

print("Label check passed.")


# =========================
# CHECK EMPTY / NULL TEXTS
# =========================
def count_invalid_texts(ds, text_col="text"):
    num_none = sum(x is None for x in ds[text_col])
    num_not_str = sum(not isinstance(x, str) and x is not None for x in ds[text_col])
    num_empty = sum(isinstance(x, str) and x.strip() == "" for x in ds[text_col])

    print(f"{text_col} -> None: {num_none}, Not string: {num_not_str}, Empty: {num_empty}")
    assert num_none == 0, "Ci sono valori None"
    assert num_not_str == 0, "Ci sono valori non stringa"
    assert num_empty == 0, "Ci sono stringhe vuote"

print("SST")
count_invalid_texts(ds_sst)

print("\nMNLI")
count_invalid_texts(ds_mnli)

print("\nText check passed.")

In [ ]:
# =========================
# SUBSAMPLE (STRATIFIED)
# =========================
import numpy as np

N_SAMPLES = 25000
SEED = 42

def stratified_subsample(ds, n_samples, seed=42):
    labels = np.array(ds["label"])
    unique_labels = np.unique(labels)

    # calcola quanti esempi per classe
    counts = {l: (labels == l).sum() for l in unique_labels}
    total = len(labels)

    target_counts = {
        l: int(n_samples * counts[l] / total)
        for l in unique_labels
    }

    # correggi eventuali arrotondamenti
    diff = n_samples - sum(target_counts.values())
    if diff > 0:
        for l in unique_labels[:diff]:
            target_counts[l] += 1

    # campiona per classe
    indices = []
    rng = np.random.default_rng(seed)

    for l in unique_labels:
        class_indices = np.where(labels == l)[0]
        sampled = rng.choice(class_indices, size=target_counts[l], replace=False)
        indices.extend(sampled)

    rng.shuffle(indices)
    return ds.select(indices)

# =========================
# APPLY
# =========================
ds_sst = stratified_subsample(ds_sst, N_SAMPLES, SEED)
ds_mnli = stratified_subsample(ds_mnli, N_SAMPLES, SEED)

In [ ]:
def dataset_summary_table(name, ds):
    return pd.DataFrame([{
        "Dataset": name,
        "Split": "train",
        "Num Samples": len(ds),
        "Columns": ", ".join(ds.column_names)
    }])

# =========================
# BUILD TABLES
# =========================
df_sst = dataset_summary_table("SST", ds_sst)
df_mnli = dataset_summary_table("MNLI", ds_mnli)

df_summary = pd.concat([df_sst, df_mnli], ignore_index=True)

# =========================
# DISPLAY SUMMARY
# =========================
display(df_summary)

# =========================
# SHOW CLEAN SAMPLES
# =========================
print("===== SST SAMPLES =====")
display(pd.DataFrame(ds_sst[:3]))

print("===== MNLI SAMPLES =====")
display(pd.DataFrame(ds_mnli[:3]))

In [ ]:
# =========================
# TRAIN / VAL / TEST SPLIT
# 80 / 10 / 10
# =========================
from sklearn.model_selection import train_test_split

SEED = 42

def split_x_y(ds, seed=42):
    texts = list(ds["text"])
    labels = list(ds["label"])

    # first split: train 80%, temp 20%
    x_train, x_temp, y_train, y_temp = train_test_split(
        texts,
        labels,
        test_size=0.2,
        random_state=seed,
        stratify=labels
    )

    # second split: val 10%, test 10%
    x_val, x_test, y_val, y_test = train_test_split(
        x_temp,
        y_temp,
        test_size=0.5,
        random_state=seed,
        stratify=y_temp
    )

    return x_train, x_val, x_test, y_train, y_val, y_test

# =========================
# MNLI
# =========================
x_mnli_train, x_mnli_val, x_mnli_test, y_mnli_train, y_mnli_val, y_mnli_test = split_x_y(ds_mnli, seed=SEED)

# =========================
# SST
# =========================
x_sst_train, x_sst_val, x_sst_test, y_sst_train, y_sst_val, y_sst_test = split_x_y(ds_sst, seed=SEED)

In [ ]:
# =========================
# PRETTY PRINT FIRST 5
# =========================
import pandas as pd

def show_first_5(x, y, name):
    df = pd.DataFrame({
        "text": x[:5],
        "label": y[:5]
    })
    print(f"\n===== {name} =====")
    display(df)

show_first_5(x_mnli_train, y_mnli_train, "MNLI TRAIN")
show_first_5(x_mnli_val, y_mnli_val, "MNLI VAL")
show_first_5(x_mnli_test, y_mnli_test, "MNLI TEST")

show_first_5(x_sst_train, y_sst_train, "SST TRAIN")
show_first_5(x_sst_val, y_sst_val, "SST VAL")
show_first_5(x_sst_test, y_sst_test, "SST TEST")

In [ ]:
# =========================
# SAVE AS PARQUET (SEPARATED)
# =========================
PROC_PATH = os.path.join(BASE_PATH, "ProcessedDB")

PATH_MNLI = os.path.join(PROC_PATH, "MNLI")
PATH_SST  = os.path.join(PROC_PATH, "SST")

# =========================
# CREATE FOLDERS
# =========================
os.makedirs(PATH_MNLI, exist_ok=True)
os.makedirs(PATH_SST, exist_ok=True)

# =========================
# MNLI
# =========================
pd.DataFrame({"text": x_mnli_train})\
    .to_parquet(os.path.join(PATH_MNLI, "X_train.parquet"), index=False)

pd.DataFrame({"label": y_mnli_train})\
    .to_parquet(os.path.join(PATH_MNLI, "y_train.parquet"), index=False)

pd.DataFrame({"text": x_mnli_val})\
    .to_parquet(os.path.join(PATH_MNLI, "X_val.parquet"), index=False)

pd.DataFrame({"label": y_mnli_val})\
    .to_parquet(os.path.join(PATH_MNLI, "y_val.parquet"), index=False)

pd.DataFrame({"text": x_mnli_test})\
    .to_parquet(os.path.join(PATH_MNLI, "X_test.parquet"), index=False)

pd.DataFrame({"label": y_mnli_test})\
    .to_parquet(os.path.join(PATH_MNLI, "y_test.parquet"), index=False)

# =========================
# SST
# =========================
pd.DataFrame({"text": x_sst_train})\
    .to_parquet(os.path.join(PATH_SST, "X_train.parquet"), index=False)

pd.DataFrame({"label": y_sst_train})\
    .to_parquet(os.path.join(PATH_SST, "y_train.parquet"), index=False)

pd.DataFrame({"text": x_sst_val})\
    .to_parquet(os.path.join(PATH_SST, "X_val.parquet"), index=False)

pd.DataFrame({"label": y_sst_val})\
    .to_parquet(os.path.join(PATH_SST, "y_val.parquet"), index=False)

pd.DataFrame({"text": x_sst_test})\
    .to_parquet(os.path.join(PATH_SST, "X_test.parquet"), index=False)

pd.DataFrame({"label": y_sst_test})\
    .to_parquet(os.path.join(PATH_SST, "y_test.parquet"), index=False)